# Quality Stream: All HCPCS per NPI across Medicare & Medicaid

**Right Problem**: Join Medicare + Medicaid on NPI to get a unified view of all procedure codes (HCPCS/CPT), volumes, and spending per provider across both payers.

**Datasets**:
- `MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv` — Medicare Physician & Other Practitioners (Provider-Service level)
- `medicaid-provider-spending.parquet` — Medicaid Provider Spending (claim-level aggregates)

**Engine**: DuckDB (in-process OLAP — queries CSV/parquet directly, no loading into memory)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import os

# Persistent local DuckDB — keeps intermediate tables across cells
con = duckdb.connect("quality_stream.duckdb")

MEDICARE_FILE = "datasets/MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv"
MEDICAID_FILE = "datasets/medicaid-provider-spending.parquet"

print(f"DuckDB {duckdb.__version__} connected.")
print(f"Medicare: {os.path.getsize(MEDICARE_FILE) / 1e9:.1f} GB")
print(f"Medicaid: {os.path.getsize(MEDICAID_FILE) / 1e9:.1f} GB")

---
## 1. Exploratory Data Analysis (EDA)

### 1.1 Medicare — Schema, Shape & Sample

In [ ]:
# Medicare: schema + shape
con.execute(f"CREATE OR REPLACE VIEW medicare_raw AS SELECT * FROM read_csv('{MEDICARE_FILE}', auto_detect=true)")

schema = con.execute("DESCRIBE medicare_raw").df()
shape = con.execute("SELECT count(*) AS rows FROM medicare_raw").fetchone()
print(f"Medicare shape: ({shape[0]:,} rows, {len(schema)} columns)\n")
print("Schema:")
display(schema)
print("\nSample (first 5 rows):")
con.execute("SELECT * FROM medicare_raw LIMIT 5").df()

### 1.2 Medicaid — Schema, Shape & Sample

In [ ]:
# Medicaid: schema + shape
con.execute(f"CREATE OR REPLACE VIEW medicaid_raw AS SELECT * FROM read_parquet('{MEDICAID_FILE}')")

schema_med = con.execute("DESCRIBE medicaid_raw").df()
shape_med = con.execute("SELECT count(*) AS rows FROM medicaid_raw").fetchone()
print(f"Medicaid shape: ({shape_med[0]:,} rows, {len(schema_med)} columns)\n")
print("Schema:")
display(schema_med)
print("\nSample (first 5 rows):")
con.execute("SELECT * FROM medicaid_raw LIMIT 5").df()

### 1.3 Null Values Analysis

In [ ]:
# Medicare: null proportions — single SQL pass
medicare_cols = con.execute("SELECT column_name FROM (DESCRIBE medicare_raw)").df()["column_name"].tolist()
null_exprs = ", ".join([f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL) / count(*), 4) AS \"{c}\"" for c in medicare_cols])
nulls_wide = con.execute(f"SELECT {null_exprs} FROM medicare_raw").df()

medicare_nulls = nulls_wide.T.reset_index()
medicare_nulls.columns = ["column", "null_pct"]
medicare_nulls = medicare_nulls.sort_values("null_pct", ascending=False).reset_index(drop=True)
print("Medicare — Null % per column:")
medicare_nulls

In [ ]:
# Medicaid: null proportions — single SQL pass
medicaid_cols = con.execute("SELECT column_name FROM (DESCRIBE medicaid_raw)").df()["column_name"].tolist()
null_exprs_med = ", ".join([f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL) / count(*), 4) AS \"{c}\"" for c in medicaid_cols])
nulls_wide_med = con.execute(f"SELECT {null_exprs_med} FROM medicaid_raw").df()

medicaid_nulls = nulls_wide_med.T.reset_index()
medicaid_nulls.columns = ["column", "null_pct"]
medicaid_nulls = medicaid_nulls.sort_values("null_pct", ascending=False).reset_index(drop=True)
print("Medicaid — Null % per column:")
medicaid_nulls

### 1.4 Data Types Summary

In [ ]:
# Side-by-side data types
mc_types = con.execute("SELECT column_name, column_type FROM (DESCRIBE medicare_raw)").df()
mc_types.columns = ["Column", "Type"]
md_types = con.execute("SELECT column_name, column_type FROM (DESCRIBE medicaid_raw)").df()
md_types.columns = ["Column", "Type"]

print("MEDICARE Data Types")
print("=" * 50)
display(mc_types)
print("\nMEDICAID Data Types")
print("=" * 50)
display(md_types)

### 1.5 Charts — Getting a Sense of the Data

In [ ]:
# Chart 1: Top 20 Provider Types by record count (Medicare)
pt_df = con.execute("""
    SELECT Rndrng_Prvdr_Type AS "Provider Type", count(*) AS "Record Count"
    FROM medicare_raw
    GROUP BY Rndrng_Prvdr_Type
    ORDER BY "Record Count" DESC
    LIMIT 20
""").df()

fig = px.bar(pt_df, x="Record Count", y="Provider Type", orientation="h",
             title="Medicare: Top 20 Provider Types by Record Count",
             color="Record Count", color_continuous_scale="Blues")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [ ]:
# Chart 2: Top 20 HCPCS codes by total services (Medicare)
hcpcs_df = con.execute("""
    SELECT HCPCS_Cd AS "HCPCS Code", sum(Tot_Srvcs)::BIGINT AS "Total Services"
    FROM medicare_raw
    GROUP BY HCPCS_Cd
    ORDER BY "Total Services" DESC
    LIMIT 20
""").df()

fig = px.bar(hcpcs_df, x="Total Services", y="HCPCS Code", orientation="h",
             title="Medicare: Top 20 HCPCS Codes by Total Services",
             color="Total Services", color_continuous_scale="Greens")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [ ]:
# Chart 3: Top 20 HCPCS codes by total claims (Medicaid)
medicaid_hcpcs_df = con.execute("""
    SELECT HCPCS_CODE AS "HCPCS Code", sum(TOTAL_CLAIMS)::BIGINT AS "Total Claims"
    FROM medicaid_raw
    GROUP BY HCPCS_CODE
    ORDER BY "Total Claims" DESC
    LIMIT 20
""").df()

fig = px.bar(medicaid_hcpcs_df, x="Total Claims", y="HCPCS Code", orientation="h",
             title="Medicaid: Top 20 HCPCS Codes by Total Claims",
             color="Total Claims", color_continuous_scale="Oranges")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [ ]:
# Chart 4: Medicare Average Payment Distribution (sampled 50K rows)
payment_df = con.execute("""
    SELECT Avg_Mdcr_Pymt_Amt
    FROM medicare_raw
    USING SAMPLE 50000
""").df()

fig = px.histogram(payment_df, x="Avg_Mdcr_Pymt_Amt", nbins=100,
                   title="Medicare: Distribution of Avg Payment Amount (sampled 50K)",
                   labels={"Avg_Mdcr_Pymt_Amt": "Avg Medicare Payment ($)"},
                   color_discrete_sequence=["#636EFA"])
fig.update_xaxes(range=[0, 500])
fig.show()

In [ ]:
# Chart 5: Medicaid Total Paid Distribution (sampled 50K rows)
medicaid_paid_df = con.execute("""
    SELECT TOTAL_PAID
    FROM medicaid_raw
    USING SAMPLE 50000
""").df()

fig = px.histogram(medicaid_paid_df, x="TOTAL_PAID", nbins=100,
                   title="Medicaid: Distribution of Total Paid (sampled 50K)",
                   labels={"TOTAL_PAID": "Total Paid ($)"},
                   color_discrete_sequence=["#EF553B"])
fig.update_xaxes(range=[0, 50000])
fig.show()

---
## 2. Data Preprocessing & Cleaning

Harmonize NPI to VARCHAR, HCPCS to uppercase VARCHAR. Aggregate metrics per NPI-HCPCS pair for each payer. All done in SQL — no Python loops.

In [ ]:
# Medicare: aggregate by NPI + HCPCS, compute total payments from avg * volume
con.execute("""
    CREATE OR REPLACE TABLE medicare_agg AS
    SELECT
        CAST(Rndrng_NPI AS VARCHAR)        AS npi,
        UPPER(TRIM(HCPCS_Cd))              AS hcpcs_code,
        -- provider info (first occurrence)
        FIRST(Rndrng_Prvdr_First_Name || ' ' || Rndrng_Prvdr_Last_Org_Name) AS provider_name,
        FIRST(Rndrng_Prvdr_Type)           AS provider_type,
        FIRST(Rndrng_Prvdr_State_Abrvtn)   AS provider_state,
        FIRST(Rndrng_Prvdr_City)           AS provider_city,
        FIRST(HCPCS_Desc)                  AS hcpcs_desc,
        -- aggregated metrics
        SUM(Tot_Benes)::BIGINT             AS medicare_total_benes,
        SUM(Tot_Srvcs)::BIGINT             AS medicare_total_srvcs,
        SUM(Tot_Bene_Day_Srvcs)::BIGINT    AS medicare_total_bene_day_srvcs,
        ROUND(SUM(Avg_Mdcr_Pymt_Amt * Tot_Srvcs), 2)   AS medicare_total_pymt,
        ROUND(SUM(Avg_Sbmtd_Chrg * Tot_Srvcs), 2)      AS medicare_total_chrg,
        ROUND(SUM(Avg_Mdcr_Alowd_Amt * Tot_Srvcs), 2)  AS medicare_total_alowd
    FROM medicare_raw
    GROUP BY 1, 2
""")

stats = con.execute("SELECT count(*) AS pairs, count(DISTINCT npi) AS npis FROM medicare_agg").fetchone()
print(f"Medicare aggregated: {stats[0]:,} NPI-HCPCS pairs from {stats[1]:,} unique NPIs")
con.execute("SELECT * FROM medicare_agg LIMIT 5").df()

In [ ]:
# Medicaid: aggregate by servicing NPI + HCPCS (across all months)
con.execute("""
    CREATE OR REPLACE TABLE medicaid_agg AS
    SELECT
        TRIM(SERVICING_PROVIDER_NPI_NUM)   AS npi,
        UPPER(TRIM(HCPCS_CODE))            AS hcpcs_code,
        SUM(TOTAL_UNIQUE_BENEFICIARIES)::BIGINT AS medicaid_total_benes,
        SUM(TOTAL_CLAIMS)::BIGINT          AS medicaid_total_claims,
        ROUND(SUM(TOTAL_PAID), 2)          AS medicaid_total_paid
    FROM medicaid_raw
    GROUP BY 1, 2
""")

stats_med = con.execute("SELECT count(*) AS pairs, count(DISTINCT npi) AS npis FROM medicaid_agg").fetchone()
print(f"Medicaid aggregated: {stats_med[0]:,} NPI-HCPCS pairs from {stats_med[1]:,} unique NPIs")
con.execute("SELECT * FROM medicaid_agg LIMIT 5").df()

---
## 3. Join: All HCPCS per NPI across Both Payers

In [ ]:
# Full outer join on NPI + HCPCS
con.execute("""
    CREATE OR REPLACE TABLE joined AS
    SELECT
        COALESCE(mc.npi, md.npi)             AS npi,
        mc.provider_name,
        mc.provider_type,
        mc.provider_state,
        mc.provider_city,
        COALESCE(mc.hcpcs_code, md.hcpcs_code) AS hcpcs_code,
        mc.hcpcs_desc,
        CASE
            WHEN mc.npi IS NOT NULL AND md.npi IS NOT NULL THEN 'Both Payers'
            WHEN mc.npi IS NOT NULL THEN 'Medicare Only'
            ELSE 'Medicaid Only'
        END AS payer_source,
        -- Medicare metrics (0 when absent)
        COALESCE(mc.medicare_total_benes, 0)          AS medicare_total_benes,
        COALESCE(mc.medicare_total_srvcs, 0)          AS medicare_total_srvcs,
        COALESCE(mc.medicare_total_pymt, 0)           AS medicare_total_pymt,
        COALESCE(mc.medicare_total_chrg, 0)           AS medicare_total_chrg,
        COALESCE(mc.medicare_total_alowd, 0)          AS medicare_total_alowd,
        -- Medicaid metrics (0 when absent)
        COALESCE(md.medicaid_total_benes, 0)          AS medicaid_total_benes,
        COALESCE(md.medicaid_total_claims, 0)         AS medicaid_total_claims,
        COALESCE(md.medicaid_total_paid, 0)           AS medicaid_total_paid
    FROM medicare_agg mc
    FULL OUTER JOIN medicaid_agg md
        ON mc.npi = md.npi AND mc.hcpcs_code = md.hcpcs_code
""")

# Summary
summary = con.execute("""
    SELECT
        count(*)              AS total_rows,
        count(DISTINCT npi)   AS unique_npis,
        count(DISTINCT hcpcs_code) AS unique_hcpcs
    FROM joined
""").fetchone()
print(f"Joined dataset: {summary[0]:,} rows | {summary[1]:,} unique NPIs | {summary[2]:,} unique HCPCS codes")

print("\nPayer source breakdown:")
display(con.execute("""
    SELECT payer_source, count(*) AS count
    FROM joined
    GROUP BY payer_source
    ORDER BY count DESC
""").df())

print("\nSample:")
con.execute("SELECT * FROM joined LIMIT 10").df()

In [ ]:
# Chart 6: Payer source breakdown — pie chart
source_df = con.execute("""
    SELECT payer_source AS "Payer Source", count(*) AS "NPI-HCPCS Pairs"
    FROM joined
    GROUP BY payer_source
""").df()

fig = px.pie(source_df, values="NPI-HCPCS Pairs", names="Payer Source",
             title="NPI-HCPCS Pair Coverage: Medicare vs Medicaid Overlap",
             color="Payer Source",
             color_discrete_map={"Medicare Only": "#636EFA", "Medicaid Only": "#EF553B", "Both Payers": "#00CC96"})
fig.show()

---
## 4. Proof: Query NPI to Get All HCPCS Codes across Both Payers

In [ ]:
# Find a provider that appears in BOTH payers with many HCPCS codes
demo = con.execute("""
    SELECT npi, count(*) AS num_codes
    FROM joined
    WHERE payer_source = 'Both Payers'
    GROUP BY npi
    ORDER BY num_codes DESC
    LIMIT 1
""").fetchone()
demo_npi = demo[0]

# Provider header
info = con.execute(f"""
    SELECT DISTINCT provider_name, provider_type, provider_state, provider_city
    FROM joined
    WHERE npi = '{demo_npi}' AND provider_name IS NOT NULL
    LIMIT 1
""").fetchone()

print(f"Demo Provider: {info[0]}")
print(f"NPI: {demo_npi}")
print(f"Type: {info[1]} | Location: {info[3]}, {info[2]}")
print("=" * 80)

# Breakdown
breakdown = con.execute(f"""
    SELECT
        count(*) AS total_hcpcs,
        count(*) FILTER (WHERE payer_source = 'Medicare Only') AS medicare_only,
        count(*) FILTER (WHERE payer_source = 'Medicaid Only') AS medicaid_only,
        count(*) FILTER (WHERE payer_source = 'Both Payers')   AS both_payers,
        sum(medicare_total_pymt) AS total_medicare_pymt,
        sum(medicaid_total_paid) AS total_medicaid_paid
    FROM joined
    WHERE npi = '{demo_npi}'
""").fetchone()

print(f"\nTotal unique HCPCS codes: {breakdown[0]}")
print(f"  - Medicare only: {breakdown[1]}")
print(f"  - Medicaid only: {breakdown[2]}")
print(f"  - Both payers:   {breakdown[3]}")
print(f"\nMedicare total payment: ${breakdown[4]:,.2f}")
print(f"Medicaid total paid:    ${breakdown[5]:,.2f}")

print(f"\nAll HCPCS codes for this provider:")
con.execute(f"""
    SELECT hcpcs_code, hcpcs_desc, payer_source,
           medicare_total_srvcs, medicare_total_pymt,
           medicaid_total_claims, medicaid_total_paid
    FROM joined
    WHERE npi = '{demo_npi}'
    ORDER BY hcpcs_code
""").df()

In [ ]:
# Reusable lookup function — query any NPI via DuckDB
def lookup_npi(npi_str):
    """Query all HCPCS codes for a given NPI across both payers."""
    npi_str = str(npi_str).strip()
    result = con.execute(f"""
        SELECT hcpcs_code, hcpcs_desc, payer_source,
               medicare_total_srvcs, medicare_total_pymt,
               medicaid_total_claims, medicaid_total_paid
        FROM joined
        WHERE npi = ?
        ORDER BY hcpcs_code
    """, [npi_str]).df()
    if len(result) == 0:
        print(f"No records found for NPI: {npi_str}")
        return None
    info = con.execute(f"""
        SELECT provider_name, provider_type, provider_state
        FROM joined WHERE npi = ? AND provider_name IS NOT NULL LIMIT 1
    """, [npi_str]).fetchone()
    if info:
        print(f"Provider: {info[0]} | Type: {info[1]} | State: {info[2]}")
    print(f"NPI: {npi_str} | Total HCPCS codes: {len(result)}")
    mc_only = (result["payer_source"] == "Medicare Only").sum()
    md_only = (result["payer_source"] == "Medicaid Only").sum()
    both = (result["payer_source"] == "Both Payers").sum()
    print(f"  Medicare only: {mc_only} | Medicaid only: {md_only} | Both: {both}")
    return result

# Example
sample_npi = con.execute("SELECT npi FROM joined LIMIT 1").fetchone()[0]
lookup_npi(sample_npi)